In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import os

from imblearn.pipeline import Pipeline
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import confusion_matrix, classification_report

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks


OUTPUT_DIR = "resultados"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def calcular_custo(cm):
    tn, fp, fn, tp = cm.ravel()
    custo = (10 * fp) + (500 * fn)
    return custo, fp, fn



def build_model(input_dim):
    model = models.Sequential([
        layers.Dense(64, input_shape=(input_dim,)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.2),

        layers.Dense(16),
        layers.BatchNormalization(),
        layers.Activation('relu'),

        layers.Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(0.001),
        loss='binary_crossentropy',
        metrics=[tf.keras.metrics.Recall(), 'accuracy']
    )
    return model


def main():

    print("Carregando treino...")
    treino = pd.read_csv('aps_failure_training_set.csv').replace('na', np.nan)

    for col in treino.columns:
        if col != 'class':
            treino[col] = pd.to_numeric(treino[col], errors='coerce')

    cols_para_remover = [
        'br_000','bq_000','bp_000','bo_000','ab_000','cr_000','bn_000','bm_000',
        'ad_000','cf_000','cg_000','ch_000','co_000','cx_000','cz_000','ct_000',
        'ak_000','db_000','cv_000','cy_000','da_000','cu_000','dc_000','ca_000',
        'dh_000','dm_000','dl_000','dk_000','eb_000','df_000','dg_000','di_000',
        'dj_000','ac_000'
    ]

    df = treino.drop(columns=cols_para_remover)

    X = df.drop(columns=['class'])
    y = df['class'].map({'neg': 0, 'pos': 1})


    print("Carregando teste...")
    teste = pd.read_csv('aps_failure_test_set.csv').replace('na', np.nan)

    for col in teste.columns:
        if col != 'class':
            teste[col] = pd.to_numeric(teste[col], errors='coerce')

    X_test = teste.drop(columns=cols_para_remover).drop(columns=['class'])
    y_test = teste['class'].map({'neg': 0, 'pos': 1})

    resultados = []


    print("\nTreinando Rede Neural...")

    imputer = KNNImputer(n_neighbors=5)
    scaler = StandardScaler()
    smote = SMOTE(random_state=42)


    X_train_real, X_val, y_train_real, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    X_tr_imp = imputer.fit_transform(X_train_real)
    X_tr_scl = scaler.fit_transform(X_tr_imp)
    X_res, y_res = smote.fit_resample(X_tr_scl, y_train_real)

    X_val_scl = scaler.transform(imputer.transform(X_val))

    model = build_model(X_res.shape[1])

    early_stop = callbacks.EarlyStopping(
    monitor='val_recall',
    mode='max',
    patience=7,
    restore_best_weights=True
    )
    model.fit(
        X_res, y_res,
        epochs=50,
        batch_size=64,
        verbose=0,
        validation_data=(X_val_scl, y_val),
        callbacks=[early_stop]
    )

    # TESTE RN
    X_test_imp = imputer.transform(X_test)
    X_test_scaled = scaler.transform(X_test_imp)

    y_pred_prob = model.predict(X_test_scaled)
    y_pred = (y_pred_prob > 0.5).astype(int)

    cm = confusion_matrix(y_test, y_pred)
    custo, fp, fn = calcular_custo(cm)

    print("\nRede Neural:")
    print(classification_report(y_test, y_pred))
    print(f"Custo: {custo}")

    np.savetxt(f"{OUTPUT_DIR}/cm_RedeNeural1.csv", cm, delimiter=",")

    resultados.append({
        "Modelo": "RedeNeural",
        "Recall_POS": classification_report(y_test, y_pred, output_dict=True)['1']['recall'],
        "F1_POS": classification_report(y_test, y_pred, output_dict=True)['1']['f1-score'],
        "Custo": custo,
        "FP": fp,
        "FN": fn
    })


if __name__ == "__main__":
    main()

Carregando treino...
Carregando teste...

Treinando Rede Neural...


/tmp/ipykernel_39540/4102133216.py:121: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  model = build_model(X_res.shape[1])


500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step

Rede Neural:
              precision    recall  f1-score   support

           0       1.00      0.97      0.98     15625
           1       0.43      0.93      0.59       375

    accuracy                           0.97     16000
   macro avg       0.71      0.95      0.79     16000
weighted avg       0.99      0.97      0.97     16000

Custo: 17140
